In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import os
os.environ["SHAPE_RESTORE_SHX"] = "YES"

# 14 - Merge RegBL and Earthquakes

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `06_OFEV_earthquakes`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Earthquakes

In [ ]:
SEISM_zone = gpd.read_file(f'{ROOT}data/raw/06_OFEV_earthquakes/Erdbebenzonen.shp').to_crs('EPSG:2056')

In [ ]:
SEISM_zone.head()

## Rename columns

In [ ]:
cols = list(SEISM_zone.columns)

# Add 'Seism - ' to the beginning of each column name except for 'geometry'
SEISM_zone.columns = ['Seism - ' + col if col != 'geometry' else col for col in cols]

### Main dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.10.csv', index_col=0)

df_regbl.head()

## Convert main DataFrame to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Merge earthquakes spatial attributes in main dataset

In [ ]:
df_regbl = df_regbl.sjoin(SEISM_zone, how='left')

## Convert back to DataFrame

In [ ]:
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.11.csv")